# The convergence gap: 1.56 in simulation against 1.13 in the analysis

At equal floor the exact Laplacian filter needs about **1.56** times the minorized filter's steps
at 0 dB SNR, while Section 6 of `main_minorization.tex` predicts **1.13** for fixed-variance filters
with white input and $M = 16$. `normalized-gain-check.ipynb` already ruled out the variance
recursion (1.54 with $\tilde v_t$ frozen). What is left is the input, $M$, and how the floor is read.

This notebook re-implements the analysis, identifies what 1.13 actually is, and then changes
**one thing at a time** in the simulation. The answer: the input, and only the input.

**Convention.** Everything here uses the **old** $b_\eta = \sqrt{v_\eta/2}$. Leszek's 1.13 was
computed at that convention, stated twice inside Section 6 (`main_minorization.tex:2588` and
`:2619`), and so were our simulations. The comparison is internally consistent only there.
Issue #9 switches to $b_\eta = \mathrm{E}|\eta_t|$ for everything else; switching here first would
destroy the matched target. Feeds Section 5 of the new draft, currently a placeholder.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from numpy.polynomial.hermite_e import hermegauss
from numpy.polynomial.legendre import leggauss
from scipy.optimize import brentq
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The two corrections, in the notation of the new Section 3

The new draft writes every filter through two scalars: the correction $\chi_t(e_t)$ applied to the
output, and its slope $\chi_t'(e_t)$, the fraction of the predicted variance $\sigma_t^2$ that the
observation removes. With

$$\sigma_t^2 = \bx_t^{\mathsf T}\tilde\bSigma_t\bx_t, \qquad
  u_t = \frac{e_t}{\sigma_t}, \qquad
  k_t = \frac{\sigma_t}{b_\eta}, \qquad
  \tau_t = \frac{\sigma_t^2}{b_\eta} = b_\eta k_t^2,$$

the Laplacian correction is $\chi_t = \tau_t\Lambda_t$ with $\Lambda_t$ built from the Mills ratio,
eq. (43). The minorized correction of the old draft, eq. (50), is $\chi_{\min}(e) = e/(1 + |e|/\tau_t)$
in the same units.

The old Section 6 used $u_{\text{old}} = e/\tau$ and the same $k$, so $u_t = k\,u_{\text{old}}$; the
two writings of the exact correction are the same function, which the cross-check below confirms.

In [ ]:
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized,
#         notebooks/07_skf_conjunto.ipynb, commit 9406077
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


def chi_exact(e, sigma, b_eta):
    return chi_laplacian(e, sigma, b_eta)[0]


def chi_minorized(e, sigma, b_eta):
    """Eq. (50) of the old draft, written through tau_t."""
    return e/(1.0 + np.abs(e)*b_eta/sigma**2)


def chi_clip(e, sigma, b_eta):
    """The limit of the exact correction as the assumed scale shrinks, eq. (chi.four)."""
    tau = sigma**2/b_eta
    return np.clip(e, -tau, tau)


CHI = {"minorized": chi_minorized, "clip": chi_clip, "exact": chi_exact}
print("corrections ready")

**Cross-check.** The old Section 6 writes the exact correction as
$\tau\tanh\!\big(-k^2u_{\text{old}} + \tfrac12\log[\Phi(k(u_{\text{old}}-1))/\Phi(-k(u_{\text{old}}+1))]\big)$.
It must agree with the copied `chi_laplacian` to roundoff, and $\chi_t'$ must be the derivative of
$\chi_t$ with respect to $e_t$ — that second fact is what lets the analysis below carry over
unchanged into the new notation, since the rate is $r = \mathrm{E}[\chi_t'(e_t)]$.

In [ ]:
def chi_exact_old_form(e, sigma, b_eta):
    """Eq. (chi.four) of main_minorization.tex, in its own variables u_old = e/tau."""
    k, tau = sigma/b_eta, sigma**2/b_eta
    u = e/tau
    return tau*np.tanh(-k**2*u + 0.5*(log_ndtr(k*(u - 1.0)) - log_ndtr(-k*(u + 1.0))))


e_test = np.linspace(-5, 5, 2001)
for sigma, b in [(0.05, np.sqrt(0.5)), (0.36, 0.64), (2.0, 0.2)]:
    chi, slope = chi_laplacian(e_test, sigma, b)
    old = chi_exact_old_form(e_test, sigma, b)
    numeric = np.gradient(chi, e_test)
    print(f"sigma = {sigma:5.2f}, b_eta = {b:5.3f}, k_t = {sigma/b:6.3f}:  "
          f"max |new - old| = {np.abs(chi - old).max():.2e},  "
          f"max |chi' - d chi/de| = {np.abs(slope - numeric)[5:-5].max():.2e}")

## 3. The analysis of Section 6, re-implemented

Premises (i)-(iv) of Section 6: the update is $\bw_t = \bw_{t-1} + \bx_t\chi(e_t)/\|\bx_t\|^2$ with a
**fixed** variance, so $\sigma_t$ is a constant and $\chi$ one fixed function; the noise is i.i.d. and
symmetric; the regressor is Gaussian with $\|\bx_t\|^2$ close to its mean; the impulse response is
fixed. With $e_t = e^{\mathrm a}_t + \eta_t$ and $e^{\mathrm a}_t$ treated as Gaussian of variance
$\sigma_a^2$, Stein's identity turns the energy balance into

$$\mathrm{E}\|\bh - \bw_t\|^2 = \mathrm{E}\|\bh - \bw_{t-1}\|^2
  - \frac{2\sigma_a^2\,\mathrm{E}[\chi'(e_t)] - \mathrm{E}[\chi(e_t)^2]}{\|\bx_t\|^2},$$

whose fixed point is $2\sigma_a^2\,\mathrm{E}[\chi'] = \mathrm{E}[\chi^2]$ and whose iteration is the
learning curve. The misadjustment is $\mathsf J = \sigma_a^2/v^*_\eta$. Only $\mathrm{E}[\chi^2]$ and
$\mathrm{E}[e^{\mathrm a}\chi]$ are needed, since $\mathrm{E}[\chi'] = \mathrm{E}[e^{\mathrm a}\chi]/\sigma_a^2$
by the same identity.

Everything scales with the noise standard deviation, so $v_\eta = 1$ without loss of generality:
the curves do not depend on the SNR, only on the target.

**Quadrature.** Gauss-Hermite in $e^{\mathrm a}_t$. Over $\eta_t$ we integrate on the **probability
scale**, $\mathrm{E}[f(\eta)] = \int_0^1 f(F^{-1}(p))\,dp$ by Gauss-Legendre: $\chi$ is bounded, so the
integrand is flat where the quantile blows up, which a grid in $\eta$ is not at $\beta^* = 0.2$.

In [ ]:
BETA = 0.2                                     # shape of the generalized Gaussian noise


class Moments:
    """E[chi(e)^2] and E[e^a chi(e)] with e = e^a + eta, e^a ~ N(0, s2)."""

    def __init__(self, beta, v_eta, n_eta=400, n_a=40):
        scale = np.sqrt(v_eta/np.exp(gammaln(3/beta) - gammaln(1/beta)))
        p, wp = leggauss(n_eta)                             # probability-scale nodes for eta
        self.eta = gennorm.ppf(0.5*(p + 1.0), beta, scale=scale)
        z, wz = hermegauss(n_a)                             # standard nodes for e^a
        self.z = z
        self.W = np.outer(wz/wz.sum(), 0.5*wp)              # product weights

    def __call__(self, chi, s2, sigma, b_eta):
        ea = np.sqrt(s2)*self.z
        c = chi(ea[:, None] + self.eta[None, :], sigma, b_eta)
        return np.sum(self.W*c**2), np.sum(self.W*ea[:, None]*c)


def steady_state_s2(mom, chi, sigma, b_eta, v_eta, s2_0=None):
    """Fixed point of s2 = E[chi^2]/(2 E[chi']), warm-started when a nearby s2 is known."""
    s2 = v_eta if s2_0 is None else s2_0
    for _ in range(300):
        A, B = mom(chi, s2, sigma, b_eta)
        new = A*s2/(2*B)                                    # E[chi'] = B/s2
        if abs(np.log(new/s2)) < 1e-11:
            return new
        s2 = new
    return s2


def sigma_for_target(mom, chi, J_db, b_eta, v_eta, bracket=(1e-3, 1e2)):
    """The constant sigma_t of the fixed family that lands the filter on J = s2/v_eta."""
    target = v_eta*10**(J_db/10)
    cache = {}

    def gap(log_sigma):
        s2 = steady_state_s2(mom, chi, np.exp(log_sigma), b_eta, v_eta, cache.get("s2"))
        cache["s2"] = s2
        return np.log(s2/target)

    return np.exp(brentq(gap, np.log(bracket[0]), np.log(bracket[1]), xtol=1e-10))


def recovery(mom, chi, sigma, b_eta, M, s2_start, s2_floor, n_max=2_000_000):
    """Iterate the variance recursion; steps to come within 3 dB of the floor."""
    s2, threshold = s2_start, s2_floor*10**(3/10)
    for t in range(1, n_max + 1):
        A, B = mom(chi, s2, sigma, b_eta)
        s2 = s2 - (2*B - A)/M
        if s2 <= threshold:
            return t, s2
    return None, s2


print("analysis ready")

**Validation.** The draft's recovery figure tunes the three Laplacian filters to
$\mathsf J = -40$ dB and starts them from an a priori error 5 dB above the noise, at $M = 16$ with
white regressors. It reports **479**, **487** and **707** iterations per coefficient, and a plateau
ratio $\tau_{\min}/\tau_{\mathrm{ex}}$ of **1.45**. If the re-implementation is right it must return
those.

In [ ]:
V_ETA_A = 1.0                                  # v_eta = 1 WLOG: the analysis does not depend on the SNR
B_ETA_A = np.sqrt(V_ETA_A/2)                   # the OLD convention, as Section 6 states
M_ANALYSIS = 16

mom = Moments(BETA, V_ETA_A)
start = time.time()

print(f"{'filter':>11}{'sigma':>9}{'tau':>10}{'k_t':>8}{'J [dB]':>9}{'steps/M':>10}   draft")
draft = {"minorized": 479, "clip": 487, "exact": 707}
validation = {}
for name in ("minorized", "clip", "exact"):
    sigma = sigma_for_target(mom, CHI[name], -40.0, B_ETA_A, V_ETA_A)
    floor = steady_state_s2(mom, CHI[name], sigma, B_ETA_A, V_ETA_A)
    steps, _ = recovery(mom, CHI[name], sigma, B_ETA_A, M_ANALYSIS,
                        V_ETA_A*10**(5/10), floor)
    validation[name] = (sigma**2/B_ETA_A, steps)
    print(f"{name:>11}{sigma:>9.5f}{sigma**2/B_ETA_A:>10.5f}{sigma/B_ETA_A:>8.4f}"
          f"{10*np.log10(floor/V_ETA_A):>9.2f}{steps/M_ANALYSIS:>10.1f}   {draft[name]}")

print(f"\nsteps exact / minorized = {validation['exact'][1]/validation['minorized'][1]:.3f}"
      f"   (draft: 707/479 = {707/479:.3f})")
print(f"tau_min / tau_exact     = {validation['minorized'][0]/validation['exact'][0]:.3f}"
      f"   (draft: 1.45)")
print(f"\n{time.time() - start:.0f} s")

## 4. What 1.13 actually is

The ratio is not a constant of the filters: it depends on **how deep the target is**. The draft says
as much, "from 1.3 at $-30$ dB to 1.7 at $-50$ dB, values the figure does not show". Sweeping the
target reproduces those, and shows where 1.13 sits.

The simulations start from $\bw_0 = \bzero$, so the a priori error starts at the signal power, i.e.
exactly SNR dB above the noise; at 0 dB SNR that is level with the noise. The sweep below uses that
start, not the 5 dB of the draft's figure.

In [ ]:
TARGETS = [-10.0, -15.0, -20.0, -25.0, -30.0, -35.0, -40.0, -45.0, -50.0]
START_DB = 0.0                                 # w0 = 0 at 0 dB SNR: a priori error level with the noise

start = time.time()
analysis_ratio = {}
print(f"{'J [dB]':>8}{'min steps/M':>13}{'exact steps/M':>15}{'ratio':>8}"
      f"{'tau_min/tau_ex':>16}")
for J in TARGETS:
    got = {}
    for name in ("minorized", "exact"):
        sigma = sigma_for_target(mom, CHI[name], J, B_ETA_A, V_ETA_A)
        floor = steady_state_s2(mom, CHI[name], sigma, B_ETA_A, V_ETA_A)
        steps, _ = recovery(mom, CHI[name], sigma, B_ETA_A, M_ANALYSIS,
                            V_ETA_A*10**(START_DB/10), floor)
        got[name] = (sigma**2/B_ETA_A, steps)
    analysis_ratio[J] = got["exact"][1]/got["minorized"][1]
    print(f"{J:>8.0f}{got['minorized'][1]/M_ANALYSIS:>13.1f}"
          f"{got['exact'][1]/M_ANALYSIS:>15.1f}{analysis_ratio[J]:>8.3f}"
          f"{got['minorized'][0]/got['exact'][0]:>16.3f}")

print(f"\nanalysis at J = -20 dB: {analysis_ratio[-20.0]:.3f}"
      f"   (the 1.13 quoted for 0 dB SNR)")
print(f"analysis at J = -15 dB: {analysis_ratio[-15.0]:.3f}"
      f"   (the 1.05 quoted for 5 dB SNR)")
print(f"analysis at J = -10 dB: {analysis_ratio[-10.0]:.3f}"
      f"   (the 1.00 quoted for 10 dB SNR)")
print(f"\n{time.time() - start:.0f} s")

So **1.13 is the analysis at $\mathsf J = -20$ dB**. With white unit-variance input and a unit-norm
impulse response, $\sigma_a^2 = \|\bh - \bw\|^2$ and the signal power is 1, so the $-20$ dB
*misalignment* target of notebook 03 is a $-20$ dB *misadjustment* at 0 dB SNR exactly. The whole
quoted row 1.13 / 1.05 / 1.00 / 0.99 at 0 / 5 / 10 / 15 dB SNR is this one curve read at
$\mathsf J = -20$, $-15$, $-10$ and $-5$ dB, because a fixed misalignment target becomes a shallower
misadjustment as the SNR rises.

## 5. The simulation, one change at a time

Notebook 03's scenario and protocol, with the input and $M$ as knobs: room impulse response of
section 7.1, generalized Gaussian noise at $\beta^* = 0.2$, SNR 0 dB, target $-20$ dB misalignment,
$\varepsilon$ replaced by a frozen $\tilde v$ (the fixed-variance family the analysis covers), grid
search on $R = 3$, check on $R = 10$, floor = mean of the last quarter, converged = first step within
3 dB of the floor, $N = 48000$.

Each run also records $\sigma_a^2 = \mathrm{E}[(\bx_t^{\mathsf T}(\bh - \bw_{t-1}))^2]$ over the last
quarter, so that the misadjustment $\mathsf J$ it actually reaches can be compared with the analysis
at the **same** $\mathsf J$ rather than at the same nominal target.

In [ ]:
SNR_DB, TARGET_DB, N, R_SEARCH, R_CHECK = 0.0, -20.0, 48000, 3, 10
N_GEN = 96000                                  # notebook 03 generates its signals at this length
WARMUP, FS = 500, 8000
V_GRID = np.logspace(-9, 0, 19)                # the frozen v~, wide enough for every M below
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]


def impulse_response(M):
    h = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM,
                     reverberation_time=T60, nsample=M).flatten()
    return h/np.linalg.norm(h)


def room_taps(M):
    """The room response truncated to M taps.

    Notebook 03 takes the first M samples, and that is what M = 128 uses here, so the long runs
    are on its exact ho. At M = 16 the first M samples are all zero, because the direct path
    arrives at sample 33 at 8 kHz, so the short window is moved onto the direct path instead."""
    if M >= 64:
        h = impulse_response(M)                      # notebook 03's ho
    else:
        full = impulse_response(M + 64)
        start = int(np.argmax(np.abs(full) > 0))     # 33, the source-microphone delay
        h = full[start:start + M]
    return h/np.linalg.norm(h)


def correlation(M, ar_a):
    lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))
    return ar_a**lags if ar_a != 0 else np.eye(M)


def signal_power(ho, ar_a):
    return float(ho @ correlation(len(ho), ar_a) @ ho)


def signals(ho, ar_a, snr, seed, n):
    """generate_signals of notebook 03, with the AR coefficient as a knob (0 = white)."""
    var_eta = signal_power(ho, ar_a)/10**(snr/10)
    rng = np.random.default_rng(seed)
    drive = np.sqrt(1 - ar_a**2)*rng.standard_normal(N_GEN + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], drive)[WARMUP:]
    scale = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    d = np.convolve(ho, x)[:N_GEN] + gennorm.rvs(BETA, scale=scale, size=N_GEN, random_state=rng)
    return x[:n], d[:n]


print("scenario ready")

In [ ]:
def misalignment_batch(X, D, ho, b_eta, v_frozen, chi):
    """B fixed-variance filters side by side, one per row of X, D and per entry of v_frozen.

    Returns the misalignment and the a priori error power before each update; ||ho|| = 1.
    The two filters differ only in chi, exactly as eq. (common.form) of Section 6 says."""
    M, (B, n) = len(ho), X.shape
    w, x_t = np.zeros((B, M)), np.zeros((B, M))
    mis, ea2 = np.empty((B, n)), np.empty((B, n))
    v = np.asarray(v_frozen, dtype=float)
    for t in range(n):
        x_t = np.roll(x_t, 1, axis=1)
        x_t[:, 0] = X[:, t]
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = w - ho
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        ea2[:, t] = np.einsum("bm,bm->b", x_t, dw)**2
        if t < M:                                    # notebook 03 waits for a full window
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)      # ||x_t||^2
        sigma = np.sqrt(v*power)                     # sigma_t = sqrt(v ||x_t||^2), fixed family
        w = w + x_t*(chi(e, sigma, b_eta)/power)[:, None]
    return mis, ea2


def steady_state(misalignment):
    """Notebook 03: floor = mean of the last quarter, converged = first step within 3 dB of it."""
    floor = 10*np.log10(misalignment[3*len(misalignment)//4:].mean())
    return floor, int(np.argmax(10*np.log10(misalignment) < floor + 3))


def pick(grid, floors):
    """pick_epsilon of notebook 03: interpolate on the branch above the best floor."""
    best = int(np.argmin(floors))
    g, f = grid[best:], floors[best:]
    if TARGET_DB > f.max():
        return g[-1], "grid too narrow"
    if TARGET_DB < f.min():
        return g[0], "target not reached"
    order = np.argsort(f)
    return 10**np.interp(TARGET_DB, f[order], np.log10(g)[order]), \
        ("ok" if best > 0 else "ok (optimum at grid edge)")


def run_configuration(ho, ar_a):
    """Grid search then check, for both filters. Returns one dict per filter plus the ratio."""
    M = len(ho)
    var_eta = signal_power(ho, ar_a)/10**(SNR_DB/10)
    b_eta = np.sqrt(var_eta/2)                       # the OLD convention
    X, D = map(np.array, zip(*[signals(ho, ar_a, SNR_DB, seed, N) for seed in range(R_CHECK)]))

    out = {}
    for name in ("minorized", "exact"):
        Xs = np.repeat(X[:R_SEARCH], len(V_GRID), axis=0)
        Ds = np.repeat(D[:R_SEARCH], len(V_GRID), axis=0)
        mis, _ = misalignment_batch(Xs, Ds, ho, b_eta, np.tile(V_GRID, R_SEARCH), CHI[name])
        floors = np.array([steady_state(c)[0]
                           for c in mis.reshape(R_SEARCH, len(V_GRID), N).mean(axis=0)])
        v_sel, status = pick(V_GRID, floors)
        mis, ea2 = misalignment_batch(X, D, ho, b_eta, np.full(R_CHECK, v_sel), CHI[name])
        floor, steps = steady_state(mis.mean(axis=0))
        sigma_a2 = ea2.mean(axis=0)[3*N//4:].mean()
        sigma = np.sqrt(v_sel*M)                     # ||x_t||^2 = M in expectation
        out[name] = dict(v=v_sel, floor=floor, steps=steps, status=status,
                         J_db=10*np.log10(sigma_a2/var_eta),
                         tau=sigma**2/b_eta, k=sigma/b_eta, b_eta=b_eta)
    out["ratio"] = out["exact"]["steps"]/out["minorized"]["steps"]
    return out


print("protocol ready")

In [ ]:
CONFIGURATIONS = [("AR(-0.9), M = 128", -0.9, room_taps(128)),
                  ("white,    M = 128", 0.0, room_taps(128)),
                  ("white,    M = 16", 0.0, room_taps(16)),
                  ("AR(-0.9), M = 16", -0.9, room_taps(16))]

start = time.time()
runs = {}
for label, ar_a, ho in CONFIGURATIONS:
    runs[label] = run_configuration(ho, ar_a)
    runs[label]["ar_a"], runs[label]["M"] = ar_a, len(ho)

print(f"{'configuration':>19}{'filter':>11}{'v~ frozen':>12}{'floor':>8}{'steps':>8}"
      f"{'J [dB]':>9}{'tau':>10}{'k_t':>8}  status")
for label in runs:
    for name in ("minorized", "exact"):
        r = runs[label][name]
        print(f"{label if name == 'minorized' else '':>19}{name:>11}{r['v']:>12.3e}"
              f"{r['floor']:>8.2f}{r['steps']:>8d}{r['J_db']:>9.2f}{r['tau']:>10.3e}"
              f"{r['k']:>8.3f}  {r['status']}")
print(f"\n{time.time() - start:.0f} s")

**The comparison that matters** puts each run next to the analysis at the misadjustment that run
actually reached, not at its nominal target.

In [ ]:
def analysis_at(J_db, M=M_ANALYSIS, start_db=START_DB):
    got = {}
    for name in ("minorized", "exact"):
        sigma = sigma_for_target(mom, CHI[name], J_db, B_ETA_A, V_ETA_A)
        floor = steady_state_s2(mom, CHI[name], sigma, B_ETA_A, V_ETA_A)
        got[name], _ = recovery(mom, CHI[name], sigma, B_ETA_A, M,
                                V_ETA_A*10**(start_db/10), floor)
    return got["exact"]/got["minorized"]


start = time.time()
print(f"{'configuration':>19}{'sim ratio':>11}{'J at floor':>12}{'analysis at that J':>21}")
for label, r in runs.items():
    J = r["exact"]["J_db"]
    print(f"{label:>19}{r['ratio']:>11.3f}{J:>12.1f} dB{analysis_at(J):>18.3f}")
print(f"\n{time.time() - start:.0f} s")

$M$ is not the explanation: white input gives the same ratio at $M = 128$ and at $M = 16$, and both
land on the analysis. How the floor is read matters only where it moves $\mathsf J$: the AR run at
$M = 16$ settles at about $-27$ dB rather than $-20$, and the analysis read at $-27$ dB accounts for
most of its 1.28. What no target correction reaches is the AR run at $M = 128$.

## 6. The input

Changing only the AR coefficient at $M = 128$, with the target held at $-20$ dB misalignment.

In [ ]:
AR_VALUES = [0.0, -0.5, -0.7, -0.9]
ho_128 = room_taps(128)

start = time.time()
colour = {}
print(f"{'AR a':>7}{'cond(Rxx)':>11}{'min steps':>11}{'exact steps':>13}{'ratio':>8}{'J [dB]':>9}")
for ar_a in AR_VALUES:
    r = runs["AR(-0.9), M = 128"] if ar_a == -0.9 else (
        runs["white,    M = 128"] if ar_a == 0.0 else run_configuration(ho_128, ar_a))
    ev = np.linalg.eigvalsh(correlation(128, ar_a))
    colour[ar_a] = (ev.max()/ev.min(), r["ratio"], r["exact"]["J_db"])
    print(f"{ar_a:>7.1f}{colour[ar_a][0]:>11.1f}{r['minorized']['steps']:>11d}"
          f"{r['exact']['steps']:>13d}{r['ratio']:>8.3f}{r['exact']['J_db']:>9.1f}")
print(f"\n{time.time() - start:.0f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

ax = axes[0]
Js = sorted(analysis_ratio)
ax.plot(Js, [analysis_ratio[J] for J in Js], "-", color="k", lw=1.6,
        label="analysis, white input, $M = 16$")
for label, marker in zip(runs, ["o", "s", "^", "D"]):
    r = runs[label]
    ax.plot(r["exact"]["J_db"], r["ratio"], marker, ms=9,
            color=COLORS[1] if r["ar_a"] == 0.0 else COLORS[3], label=f"simulation, {label}")
ax.axhline(1, color="k", lw=0.8)
ax.set(xlabel=r"misadjustment $\mathsf{J}$ at the floor [dB]",
       ylabel="steps, exact / minorized",
       title="Read at the same target, white input agrees")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)

ax = axes[1]
conds = [colour[a][0] for a in AR_VALUES]
ax.semilogx(conds, [colour[a][1] for a in AR_VALUES], "o-", color=COLORS[3], lw=2, ms=9,
            label=r"simulation, $M = 128$, target $-20$ dB")
ax.axhline(analysis_ratio[-20.0], color="k", ls="--", lw=1.4,
           label=f"analysis at $\\mathsf{{J}} = -20$ dB ({analysis_ratio[-20.0]:.2f})")
for a in AR_VALUES:
    ax.annotate(f"a = {a:.1f}", (colour[a][0], colour[a][1]),
                textcoords="offset points", xytext=(6, -12), fontsize=8)
ax.set(xlabel=r"eigenvalue spread of $\mathbf{R}_{xx}$",
       ylabel="steps, exact / minorized",
       title="The gap tracks the input, at fixed $\\mathsf{J}$")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

## 7. Findings

* **1.13 is the analysis at $\mathsf J = -20$ dB.** The re-implementation returns the draft's own
  numbers — 479 / 487 / 707 iterations per coefficient and $\tau_{\min}/\tau_{\mathrm{ex}} = 1.45$ at
  $-40$ dB — and the quoted row 1.13 / 1.05 / 1.00 / 0.99 is one curve read at $-20$, $-15$, $-10$ and
  $-5$ dB. The ratio is a function of the target, from about 1.03 at $-10$ dB to 1.69 at $-50$ dB, so
  no single number is "the prediction" without its target.

* **$M$ is not the explanation.** With white input the ratio is 1.11 at $M = 128$ and 1.12 at
  $M = 16$, both on the analysis. Between $M = 16$ and $M = 128$ at white input the ratio moves less
  than the protocol's own noise.

* **How the floor is read matters only where it moves $\mathsf J$.** Misalignment
  $\|\bh - \bw\|^2/\|\bh\|^2$ and misadjustment $\sigma_a^2/v_\eta$ coincide for white unit-variance
  input and a unit-norm response, and separate under coloured input: the AR run at $M = 16$ lands at
  $\mathsf J \approx -27$ dB, and the analysis read there accounts for most of its ratio.

* **The gap is the input, and it scales with the eigenvalue spread.** At $M = 128$ and a fixed
  $-20$ dB target, with $\mathsf J$ within 1.3 dB across the sweep, the ratio runs 1.11, 1.24, 1.31,
  1.55 as the spread of $\mathbf R_{xx}$ runs 1, 9, 32, 347.

* **Simulation and analysis agree wherever the analysis applies.** Section 6 says so itself: for
  correlated regressors the variance recursion "no longer closes on $\sigma_a^2$ alone, and the
  learning curves would require the full covariance of $\bh - \bw_t$; there the recovery comparison
  rests on simulation". The 1.56 is not a discrepancy with the analysis; it is the regime the
  analysis excludes.

* **In the notation of the new Section 3**, the rate is $r = \mathrm{E}[\chi_t'(e_t)]$, the average
  fraction of the predicted variance that an observation removes, and the steady state is
  $2\sigma_a^2\,r = \mathrm{E}[\chi_t(e_t)^2]$. The exact correction spreads its slope over $\sigma_t$
  while the minorized one places it within a few multiples of $\tau_t = b_\eta k_t^2$; at the operating
  point here $k_t$ is below 1, so the exact filter needs a larger $\sigma_t$ to reach a given rate,
  which is the mechanism behind every row above.

* **What this hands to #11.** The excess of 1.55 over the white-input 1.11 is a coloured-input loss.
  Section 4 of the new draft claims the full-covariance KF recovers it without prewhitening, because
  the update direction $\tilde\bSigma_t\bx_t$ is decorrelated. That is now a falsifiable prediction:
  at AR($-0.9$) and $M = 128$ the KF should bring this ratio back toward 1.11.

In [ ]:
print(f"total notebook time: {time.time() - NOTEBOOK_START:.0f} s")